# S6E11 | The 0.95330 Ladder, One Step Further | LB 0.95333

**Our earlier versions** (all in the version history): v13 0.95330, v12 0.95300, v11 0.95291, v10 0.95288, v9 0.95239.

This version scores **0.95333** on the public LB. It starts from the frozen Strike 9 file by Moran Z (0.95330) and takes one small step in logit-rank space away from two public members that sit almost on top of it. Read **How to read the gaps** before you use it as a final pick: the step wins on the public LB and loses on CV.

## What it does

1. Loads three public files and checks each against its SHA-256:
   - `submission_095330.csv`, the Strike 9 file from [S6E11 Strike9 LB 0.95330](https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330) by Moran Z.
   - `test_gbn_bag5_w512.npy` (MLP: residual MLPs on a GLM's logit, 5 seeds) and `test_gb_bag5.npy` (GB: 4-leaf LightGBM boosted from a GLM, 5 seeds) from the [S6E11 OOF Bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) by goodpjw2008.
2. Turns each into logit ranks, `logit((rank - 0.5) / n)`. AUC only sees order, so this puts all three on one scale.
3. Steps away from the two members: `score = 1.15 F - 0.10 MLP - 0.05 GB`, which is `F + 0.10 (F - MLP) + 0.05 (F - GB)`.
4. Ranks the result, writes `submission.csv` and checks its SHA-256 against the file that scored 0.95333.

## How we got here: probe table

Every probe starts from the Strike 9 file `F`, in logit ranks.

| Probe | Public LB |
|---|---|
| F unchanged (v13) | 0.95330 |
| F + 0.15 (F - our 0.95300 backbone) | 0.95330 |
| 0.90 F + 0.10 MLP | 0.95326 |
| F + 0.05 (F - megayak shallow-tree) | 0.95330 |
| F + 0.05 (F - MLP) | 0.95332 |
| F + 0.10 (F - MLP) | 0.95332 |
| F + 0.15 (F - MLP) | 0.95332 |
| **F + 0.10 (F - MLP) + 0.05 (F - GB) (this notebook)** | **0.95333** |
| F + 0.10 (F - MLP) + 0.10 (F - GB) | 0.95333 |

- **Adding the MLP hurts, and stepping away from it helps.** That matches Moran's own finding that collinear tree bags dilute the language-model backbone (his Strike 5A lost 0.00008).
- **The gain flattens fast.** 0.05, 0.10 and 0.15 away from the MLP all score 0.95332, and a second step away from GB adds one more point.

## CV evidence: the step does not hold up

F has no OOF, so we test the same steps on the nested OOF of goodpjw2008's Part 12 wide LM stack (AUC 0.952565), the closest public stand-in:

| Step on the wide-stack OOF | CV change |
|---|---|
| 0.10 away from MLP | -4.5e-5 |
| 0.10 away from MLP + 0.05 away from GB (this notebook) | -7.4e-5 |
| 0.10 away from MLP + 0.10 away from GB | -10.8e-5 |

Every step away loses on CV, and the bigger the step, the more it loses. So the +0.00003 on the public LB is most likely fitted to the public 20%, not a better model.

## How to read the gaps

- The public LB is about 20% of the test set, roughly 60k rows. 0.00001 is noise and 0.00003 is barely more.
- F already contains LB-gradient steps by its author, and this notebook adds one more. Both can fade on the private 80%.
- **For your two final picks, treat this as the best-public pick and pair it with a CV-backed one**, such as our v12 (0.95300).
- The next real gain is new models with different errors, checked on OOF first, not more blending.

## Credits

- **Moran Z**, [S6E11 | Grandmaster Ladder | LB 0.95330](https://www.kaggle.com/code/moranzavdi/s6e11-grandmaster-ladder-lb-0-95330) and the [Strike9 dataset](https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330): the base file. Almost all of this score is his work.
- **goodpjw2008**, the [S6E11 OOF Bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) (the MLP and GB predictions used here), [S6E11 Part 12: Wide Stack + 2nd LM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298) and [s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts): the wide LM stack and the OOFs behind every CV check.
- **megayak**, [S6E11 | Per-Value + Digit TE | Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack): the boosting-from-a-GLM recipe behind GB, and the shallow-tree file probed above.
- **Malyshev Danil**, [Fraud Signal Lab | GPT-2 and GLM Fusion](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion): the 65/35 backbone of our earlier versions.
- **masayakawamata**, [GPT-2 Target-First Classifier](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500), and **Chris Deotte**, [S6E10 1st place writeup](https://www.kaggle.com/competitions/playground-series-s6e10/writeups/1st-place-tfms-and-llms-with-tabular-data): the target-first GPT-2 idea behind the language models.
- **alexchilton** ([Shallow-Tree Stack](https://www.kaggle.com/code/alexchilton/credit-card-fraud-lb-0-9524-shallow-tree-stack)) and **amanatar** ([Sovereign Fusion](https://www.kaggle.com/code/amanatar/s6e11-sovereign-fusion-solution)): probed in earlier versions, not used.
- The wide stack pools public predictions from Sadam Torres, alexchilton, Malyshev Danil, lazyyeou, megayak, Will, queyrusi, Vladimir Demidov, Nilanjan_Saha123, masayakawamata, Sameer Shaikh and goodpjw2008.

If this helps, please upvote the source notebooks too, starting with Moran Z.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.special import logit

# Public inputs, checked byte for byte
EXPECTED = {
    'submission_095330.csv': 'f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105',  # Strike 9 by Moran Z
    'test_gbn_bag5_w512.npy': '035d6544d7c03108fc42ee296a8660a45825e3b1b19b7cabce1c3f42cac4e7cb',  # goodpjw2008 OOF bank
    'test_gb_bag5.npy': '7d54e5f17e0707d88fec1a2cccb05885978ae6cc2a595c5f4575c7b7d45a9bb8',  # goodpjw2008 OOF bank
}
OUT_SHA = '499c956edec2fbd2cd8246239b48a928555f1f054217149ee6ae87f9c1760e88'  # the file that scored 0.95333
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/submission.csv')

def locate(name):
    hits = sorted(INPUT.rglob(name))
    assert hits, name
    return hits[0]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def logit_rank(v):
    return logit((rankdata(v) - 0.5) / len(v))

paths = {name: locate(name) for name in EXPECTED}
for name, path in paths.items():
    assert sha256(path) == EXPECTED[name], (name, sha256(path))

sample = pd.read_csv(locate('sample_submission.csv'))
strike9 = pd.read_csv(paths['submission_095330.csv'])
assert (strike9['id'].to_numpy() == sample['id'].to_numpy()).all()

F = logit_rank(strike9['is_fraud'].to_numpy())
MLP = logit_rank(np.load(paths['test_gbn_bag5_w512.npy']))
GB = logit_rank(np.load(paths['test_gb_bag5.npy']))
assert len(F) == len(MLP) == len(GB) == len(sample)

# One step away from two members that sit almost on top of Strike 9
# 1.15 F - 0.10 MLP - 0.05 GB  =  F + 0.10 (F - MLP) + 0.05 (F - GB)
score = 1.15 * F - 0.10 * MLP - 0.05 * GB
print('correlation with Strike 9:', round(float(np.corrcoef(score, F)[0, 1]), 6))

sample.assign(is_fraud=(rankdata(score) - 0.5) / len(score)).to_csv(OUT, index=False)
assert sha256(OUT) == OUT_SHA, sha256(OUT)
print('wrote', OUT, len(sample), 'rows, sha', OUT_SHA[:16])

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(sub.shape)
print(sub.head())
print(sub['is_fraud'].describe())